# Text cleaning  

Raw review text is mesy, may contain HTML tags, URLs, numbers, and inconsistent formatting. Standard cleaning pipelines often destroy sentiment context by removing negations: "not good" becomes "good", completely changing the meaning.

  This pipeline addresses that by handling negations before any other cleaning step, preserving context linke "not_good" and "not_clean" as single tokens.

In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
import re
from tqdm.auto import tqdm
tqdm.pandas()

INFO: Pandarallel will run on 6 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [2]:
df = pd.read_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\airbnb_english.csv")
df.head()

,date,reviewer_name,comments,language
0,3/30/2009,Lam,Daniel is really cool. The place was nice and ...,en
1,7/9/2012,Gregory,If you want the authentic Amsterdam houseboat ...,en
2,7/15/2012,Michael,Unique and luxurious to be sure. I couldn't re...,en
3,4/24/2009,Alice,Daniel is the most amazing host! His place is ...,en
4,8/12/2012,Brian,My wife and I recently stopped in Amsterdam fo...,en


## Feature Extraction

Before cleaning the text, we extract numerical features that capture emotional signals like exclamation marks, question marks, capitalization ratio and word count. These features are extracted from the raw text because the cleaning process would remove them.

In [3]:
def extract_features(df):
    df['exclamation_count'] = df['comments'].str.count('!')
    df['question_count'] = df['comments'].str.count('\?')
    df['caps_ratio'] = df['comments'].apply(lambda x: sum(1 for c in str(x) if c.isupper())/max(len(str(x)),1))
    df['word_count'] = df['comments'].str.split().str.len()
    return df

df= extract_features(df)
df[['comments', 'exclamation_count', 'question_count', 'caps_ratio', 'word_count']].head()

,comments,exclamation_count,question_count,caps_ratio,word_count
0,Daniel is really cool. The place was nice and ...,2,0,0.036000,46
1,If you want the authentic Amsterdam houseboat ...,2,0,0.013230,306
2,Unique and luxurious to be sure. I couldn't re...,3,0,0.018280,167
3,Daniel is the most amazing host! His place is ...,2,0,0.011976,58
4,My wife and I recently stopped in Amsterdam fo...,0,0,0.017937,120


Cleaning functions

In [4]:
def handle_negations(text):
    text = re.sub(r"won't","will not", text)
    text = re.sub(r"can't","can not", text)
    text = re.sub(r"n't","not", text)
    text = re.sub(r"\bnot\s+(\w+)\s+(\w+)",r"not_\1_\2",text)
    text = re.sub(r"\bdidnt\b","did not",text)
    text = re.sub(r"\bdont\b","do not",text)
    text = re.sub(r"\bisnt\b","is not",text)
    text = re.sub(r"\bcant\b","cannot",text)
    text = re.sub(r"\bdoesnt\b","does not",text)
    return text

def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

def remove_mention_hashtags(text):
    return re.sub(r'\@\w+|\#\w+', '', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def handle_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def clean_html(text):
    return re.sub('<.*?>', ' ', text)

tag_dict = {
    'J': wordnet.ADJ,
    'N': wordnet.NOUN,
    'V': wordnet.VERB,
    'R': wordnet.ADV
}

def get_word_net(tag):
    return tag_dict.get(tag[0].upper(), wordnet.NOUN)

lemmatizer = nltk.stem.WordNetLemmatizer()

def lemmitization(text):
    words = text.split()
    tagged_words = pos_tag(words)
    return ' '.join([
        lemmatizer.lemmatize(word, get_word_net(tag)) for word, tag in tagged_words
    ])

def clean_text(text):
    text = str(text).lower()
    text = clean_html(text)
    text = remove_urls(text)
    text = remove_mention_hashtags(text)
    text = handle_negations(text)
    text = remove_numbers(text)
    text = handle_whitespace(text)
    text = lemmitization(text)
    return text

In [5]:
#verifying if the whole functions block works
ejemplo = df['comments'].iloc[0]
print("ORIGINAL:")
print(ejemplo)
print()
print("LIMPIO:")
print(clean_text(ejemplo))

ORIGINAL:
Daniel is really cool. The place was nice and clean. Very quiet neighborhood. He had maps and a lonely planet guide book in the room for you to use. I didnt have any trouble finding the place from Central Station. I would defintely come back! Thanks!

LIMPIO:
daniel be really cool. the place be nice and clean. very quiet neighborhood. he have map and a lonely planet guide book in the room for you to use. i do not have any trouble find the place from central station. i would defintely come back! thanks!


In [7]:
#Note: Pandarallel was tested to speed up processing across the available CPU cores,
#but it presents compatibility issues on Windows due to its inability to share
#function scope between processes. progress_apply was used as an alternative.
df['cleaned_comments'] = df['comments'].progress_apply(clean_text)

  0%|          | 0/258347 [00:00<?, ?it/s]

In [8]:
pd.set_option('display.max_colwidth', None)
df[['comments','cleaned_comments']].sample(5)

,comments,cleaned_comments
200336,Really a lovely place! Good and friendly service.,really a lovely place! good and friendly service.
245261,Very cozy and charming little house right on a gram and river. Great location far enough away from Amsterdam to relax but close enough to get there in ~10 mins by Uber. The small town nearby was also very quaint and cute. We rented linens and towels from Amber which was definitely worth it since we were trying to pack light. Nice area to bike or take an evening stroll.,very cozy and charm little house right on a gram and river. great location far enough away from amsterdam to relax but close enough to get there in ~ min by uber. the small town nearby be also very quaint and cute. we rent linen and towel from amber which be definitely worth it since we be try to pack light. nice area to bike or take an evening stroll.
209177,"Perfect area for a nice stay in the Jordaan , host has excellent communication!","perfect area for a nice stay in the jordaan , host have excellent communication!"
108727,"By far the best airbnb I have stayed in. I am serious, if you are visiting Amsterdam with a couple of friends, this is your dream house. I don’t know where to begin... We have communicated with Douwe almost daily, they have sent us a deeply informative guide about their house and surroundings. The house is located conveniently, you can walk to the most attractive parts of the city from this house. It is far away from the touristic part of the city which makes you feel like a local even more. The house itself had everything in it, the hosts left some beers in the fridge, snacks, personal hygiene materials in the toilets. The level of comfort is astonishing, the details in this house made me want to stay for another 5 weeks. It was clean and also affordable price for what you are getting. Do yourselves a favour and stay here, but please take care of the house because it is a special, cozy, inspiring place to live in. I will always remembers the moments I have stayed here. Thank you for everything Melissa & Douwe.","by far the best airbnb i have stay in. i be serious, if you be visit amsterdam with a couple of friends, this be your dream house. i don’t know where to begin... we have communicate with douwe almost daily, they have send u a deeply informative guide about their house and surroundings. the house be locate conveniently, you can walk to the most attractive part of the city from this house. it be far away from the touristic part of the city which make you feel like a local even more. the house itself have everything in it, the host leave some beer in the fridge, snacks, personal hygiene material in the toilets. the level of comfort be astonishing, the detail in this house make me want to stay for another weeks. it be clean and also affordable price for what you be getting. do yourselves a favour and stay here, but please take care of the house because it be a special, cozy, inspire place to live in. i will always remember the moment i have stay here. thank you for everything melissa & douwe."
32541,地理位置和central station很近。阿姆斯特丹其实很小基本的地点走走都可以到。Stijn非常友好:) 住在船上是一种很特别的体验，很舒服，很有趣。特别是坐在驾驶室，晒太阳的感觉真是超级棒:) \r<br/>\r<br/>Both the location and host are great. It is a very local way to explore Amsterdam. We miss the days in Stijn' house:),地理位置和central station很近。阿姆斯特丹其实很小基本的地点走走都可以到。stijn非常友好:) 住在船上是一种很特别的体验，很舒服，很有趣。特别是坐在驾驶室，晒太阳的感觉真是超级棒:) both the location and host be great. it be a very local way to explore amsterdam. we miss the day in stijn' house:)


Since some comments have diferent characters we are going to use a function that identifies how many characters are english ascii and check the diference and those that have more than 90% of ascii characters will remain the others will be filtered out.

In [9]:
def is_mostly_english(text):
    ascii_chars = sum(1 for c in str(text) if ord(c) < 128)
    total_chars = max(len(str(text)),1)
    ratio = ascii_chars/ total_chars
    return ratio > 0.90

df_english = df[df['cleaned_comments'].apply(is_mostly_english)]
print("Lines after filter:",len(df_english))


Lines after filter: 258002


In [12]:
df_english.head()

,date,reviewer_name,comments,language,exclamation_count,question_count,caps_ratio,word_count,cleaned_comments
0,3/30/2009,Lam,Daniel is really cool. The place was nice and clean. Very quiet neighborhood. He had maps and a lonely planet guide book in the room for you to use. I didnt have any trouble finding the place from Central Station. I would defintely come back! Thanks!,en,2,0,0.036000,46,daniel be really cool. the place be nice and clean. very quiet neighborhood. he have map and a lonely planet guide book in the room for you to use. i do not have any trouble find the place from central station. i would defintely come back! thanks!
1,7/9/2012,Gregory,"If you want the authentic Amsterdam houseboat experience, this is it! It is a great boat, located on a quiet canal with a view of the Rijksmuseum towers. Spotlessly clean, comfortable bed, hot shower, good wifi signal, nice kitchen with fridge, cofee maker, and stove for cooking meals. The location is very accessible to museums, gym bar and bakery on the corner, nine streets area with lots of resaraunts and bars and jazz clubs, Vondelpark, shopping on Utrechtstraat, and all of the sites of central Amsterdam. \r<br/>\r<br/>Now for the special part: If you are at all interested in boats and maritime history, this one is a great find. The cabin where guests stay is actually the cargo hold within the original hull of a freight boat that once hauled goods all over Holland. (Just imagine the countless tons of cargo that came in and out of the place where you are sleeping!) And yet the interior beautifully and comfortably remodeled using new and salvaged materials including a number of hatches that serve as skylights. The original steerhouse is still intact, along with the original one-cylinder engine down below. But the highlight is the captain's quarters where the family who owned and operated the boat lived during the 1920s (not open for guests to sleep in, yet). Stepping down into the quarters, complete with original cabinets and cookstove, feels like traveling back in time. The host cares a lot about preserving the historical character of his boat, because most other boats like it have long since gone to the scrapyard. He showed us fascinating photographs of the original owners who lived on the boat when it plied the canals nearly 100 years ago. \r<br/>\r<br/>There are many houseboats in Amsterdam. A lot of them are glorified trailer houses on flat barges. This one is a gem.",en,2,0,0.013230,306,"if you want the authentic amsterdam houseboat experience, this be it! it be a great boat, locate on a quiet canal with a view of the rijksmuseum towers. spotlessly clean, comfortable bed, hot shower, good wifi signal, nice kitchen with fridge, cofee maker, and stave for cook meals. the location be very accessible to museums, gym bar and bakery on the corner, nine street area with lot of resaraunts and bar and jazz clubs, vondelpark, shopping on utrechtstraat, and all of the site of central amsterdam. now for the special part: if you be at all interested in boat and maritime history, this one be a great find. the cabin where guest stay be actually the cargo hold within the original hull of a freight boat that once haul good all over holland. (just imagine the countless ton of cargo that come in and out of the place where you be sleeping!) and yet the interior beautifully and comfortably remodel use new and salvaged material include a number of hatch that serve a skylights. the original steerhouse be still intact, along with the original one-cylinder engine down below. but the highlight be the captain's quarter where the family who own and operate the boat live during the s (not_open_for guest to sleep in, yet). step down into the quarters, complete with original cabinet and cookstove, feel like travel back in time. the host care a lot about preserve the historical character of his boat, because most other boat like it have long since go to the scrapyard. he show u fascinate photograph o

## Saving the dataset

  The balanced dataset is saved as a CSV file to be used in the next notebook for sentiment labeling.

In [10]:
df_english.to_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\airbnb_english_cleaned.csv", 
                  index= False)